# UMT-ViT — package-driven notebook (milestone U6)

**Universal Multi-Scale Topographic Vision Transformer.** This is the
*package-driven* twin of the canonical `kaggle_umtvit.ipynb`: the exact same
self-supervised experiment — dual-scale cross-attention ViT → 3-D latent voxel
volume → differentiable 3-D SOM — but every stage is imported from the
installed **`umtvit`** package rather than defined inline. Same visuals, same
artifacts, same `umtvit_web.json` (schema v1). See `notebooks/README.md` for
how the three notebooks relate.

**Honesty notes.** Training is label-free; labels enter *only* the evaluation
probes. The latent volume's Z-axis is a **learned representational hierarchy**,
not physical/anatomical depth — whether scale ordering emerged is *measured*
(Z-axis probe) and reported, never assumed. The frozen-feature probe / k-NN are
SSL yardsticks, **not** comparable to supervised end-to-end accuracies.

## How to swap datasets
Edit one string in the config cell: `CONFIG_NAME = "shapes"` (the CPU/CI
default) → `"ham10000"` or `"eurosat"`. Each name loads
`configs/<name>.yaml` through `umtvit.load_config`. For HAM10000 on Kaggle the
metadata CSV + image dirs are auto-detected under `/kaggle/input`. An optional
`OVERRIDES` dict tweaks any field (e.g. a shorter epoch budget) before
validation.

In [ ]:
# ── setup: install/locate the umtvit package · imports · seeds · device ──
import glob as _glob
import importlib
import os
import subprocess
import sys
import warnings
from pathlib import Path


def _find_package_dir():
    """Locate the editable `umtvit` source tree (experiments/umtvit)."""
    seen, candidates = set(), []
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates += [base / "experiments" / "umtvit", base, base.parent]
    # Kaggle mounts the repo under a variable path; glob for the package root.
    for pat in ("/kaggle/**/experiments/umtvit/pyproject.toml",
                "/kaggle/**/umtvit/pyproject.toml"):
        candidates += [Path(p).parent for p in _glob.glob(pat, recursive=True)]
    for cand in candidates:
        cand = cand.resolve()
        if cand in seen:
            continue
        seen.add(cand)
        if (cand / "pyproject.toml").is_file() and (cand / "umtvit" / "__init__.py").is_file():
            return cand
    return None


def _ensure_umtvit():
    """Import `umtvit`, installing it editable (or falling back to sys.path)."""
    try:
        import umtvit  # already importable (pre-installed / same-dir)
        return umtvit, "pre-installed"
    except ImportError:
        pass
    pkg_dir = _find_package_dir()
    if pkg_dir is None:
        raise RuntimeError(
            "could not locate the umtvit package (experiments/umtvit). On Kaggle, "
            "attach the repo; locally, run this notebook from within the checkout.")
    # Prefer `pip install -e` so `umtvit` is a real, importable distribution.
    # pip's own chatter is captured so the notebook's stderr stays clean; any
    # failure falls back to a sys.path insertion of the source tree.
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", str(pkg_dir),
             "--no-build-isolation", "--no-deps", "-q"],
            check=True, capture_output=True, text=True)
        importlib.invalidate_caches()
        import umtvit
        return umtvit, f"pip install -e {pkg_dir}"
    except Exception:
        sys.path.insert(0, str(pkg_dir))
        importlib.invalidate_caches()
        import umtvit
        return umtvit, f"sys.path fallback ({pkg_dir})"


umtvit, _install_mode = _ensure_umtvit()
CONFIG_DIR = Path(umtvit.__file__).resolve().parent.parent / "configs"

import contextlib
import io
import json
import math
import random
import time

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.lines import Line2D
from PIL import Image
from IPython.display import HTML, display

from umtvit import load_config
from umtvit.data import UniversalDataset
from umtvit.models import UMTViT, Soft3DSOM
from umtvit.engine import Trainer, AblationRunner, ABLATIONS
from umtvit.eval import (
    run_evaluation, render_report, radial_spectrum, spectral_centroid, zaxis_probe)

# Notebook hygiene: keep visual/library warnings out of the output stream (this
# is a demo notebook; the package itself is exercised under -W error by pytest).
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 90, "axes.grid": False})


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"umtvit {umtvit.__version__} · {os.path.dirname(umtvit.__file__)} · {_install_mode}")
print(f"configs: {CONFIG_DIR} · device: {DEVICE} · torch {torch.__version__}")

## Config — the only cell you edit

`CONFIG_NAME` picks `configs/<name>.yaml`; `load_config` parses **and
validates** it (the config is the single source of truth). `OVERRIDES` is an
optional dotted-path patch applied before re-validation — handy for a quick
run (fewer epochs) without touching the YAML. `VIZ` holds notebook-only display
knobs (how many probe images to track, embedding-subset size, animation fps)
that live outside the run schema.

In [ ]:
# ════════════════ CONFIG ════════════════
CONFIG_NAME = "shapes"          # "shapes" (CPU/CI default) | "ham10000" | "eurosat"

OVERRIDES = {}                  # dotted-path patches, e.g. {"train.epochs": 1}
# OVERRIDES = {"train.epochs": 1}          # ◀ quick smoke run

VIZ = {                         # notebook-only display knobs (not in the schema)
    "probe_images": 6,          # images whose latent cubes we track over training
    "embed_subset": 400,        # points in the embedding animation
    "anim_fps": 6, "max_anim_frames": 60,
}


def resolve_ham10000_paths(cfg):
    """Locate HAM10000's metadata CSV + image dirs under /kaggle/input.

    Kaggle mirrors this dataset under varying paths; rather than hard-code them
    we glob. Only runs for the ham10000 config when the configured paths are
    absent, so the shapes CI default never triggers filesystem globbing.
    """
    ds = cfg.dataset
    csv_ok = bool(ds.metadata_csv) and os.path.isfile(ds.metadata_csv)
    dirs = ds.image_dir if isinstance(ds.image_dir, list) else (
        [ds.image_dir] if ds.image_dir else [])
    dirs_ok = bool(dirs) and all(os.path.isdir(d) for d in dirs)
    if csv_ok and dirs_ok:
        return cfg                                        # configured paths exist
    csvs = sorted(_glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True))
    if not csvs:
        csvs = sorted(_glob.glob("/kaggle/input/**/*metadata*.csv", recursive=True))
    if not csvs:
        raise FileNotFoundError(
            "HAM10000_metadata.csv not found under /kaggle/input/ — attach the "
            "kmader/skin-cancer-mnist-ham10000 dataset, or set dataset.metadata_csv/"
            "image_dir in configs/ham10000.yaml.")
    ds.metadata_csv = csvs[0]
    base = os.path.dirname(csvs[0])
    found = [p for p in sorted(_glob.glob(os.path.join(base, "*")))
             if os.path.isdir(p) and any(f.endswith(".jpg") for f in os.listdir(p)[:20])]
    ds.image_dir = found or [base]
    print("HAM10000 metadata:", ds.metadata_csv)
    print("HAM10000 image dirs:", ds.image_dir)
    return cfg


def apply_overrides(cfg, overrides):
    """Set each ``dotted.path = value`` on the config, then re-validate."""
    for dotted, value in overrides.items():
        parts = dotted.split(".")
        target = cfg
        for part in parts[:-1]:
            target = getattr(target, part)
        if not hasattr(target, parts[-1]):
            raise KeyError(f"OVERRIDES: unknown config path {dotted!r}")
        setattr(target, parts[-1], value)
    cfg.model.image_size = None   # re-derive from dataset.image_size on validate()
    return cfg.validate()


cfg = load_config(CONFIG_DIR / f"{CONFIG_NAME}.yaml")
if CONFIG_NAME == "ham10000":
    resolve_ham10000_paths(cfg)   # auto-detect on Kaggle; no-op elsewhere
if OVERRIDES:
    cfg = apply_overrides(cfg, OVERRIDES)

set_seed(cfg.train.seed)
m = cfg.model
print(f"dataset={cfg.dataset.name} · loader={cfg.dataset.loader} · "
      f"image={cfg.dataset.image_size}px · L(depth)={m.depth} · "
      f"volume={m.volume_h}x{m.volume_w}x{m.depth}x{m.volume_channels} · "
      f"SOM={tuple(m.som_grid)} · epochs={cfg.train.epochs} · batch={cfg.train.batch_size}")

## Stage 1 — Universal data pipeline

`umtvit.data.UniversalDataset(cfg, split, mode)` reads the whole pipeline from
the config: loader (shapes / imagefolder / csv), leakage-free grouped splits,
and the named augmentation policy. `two_view` yields the two contrastive views;
`eval` yields a deterministic resize-only view for the probes.

In [ ]:
# ── build datasets + loaders; look at what the model will see ────────────
train_ds = UniversalDataset(cfg, "train", "two_view")
eval_train_ds = UniversalDataset(cfg, "train", "eval")
eval_test_ds = UniversalDataset(cfg, "test", "eval")

CLASSES = train_ds.classes
NUM_CLASSES = len(CLASSES) if cfg.dataset.label_column else 0

print(f"train={len(train_ds)} · eval-train={len(eval_train_ds)} · "
      f"eval-test={len(eval_test_ds)} · "
      f"classes={CLASSES if NUM_CLASSES else 'unlabeled'}")

from torch.utils.data import DataLoader
_peek = DataLoader(train_ds, batch_size=cfg.train.batch_size, shuffle=True)
va, vb, yy = next(iter(_peek))
n_peek = min(6, va.shape[0])
fig, axes = plt.subplots(2, n_peek, figsize=(2 * n_peek, 4.2), squeeze=False)
fig.suptitle(f"two augmented views per image — policy '{cfg.dataset.augmentation}'")
for j in range(n_peek):
    for row, batch in enumerate((va, vb)):
        ax = axes[row, j]
        ax.imshow(batch[j].permute(1, 2, 0).numpy())
        ax.set_axis_off()
        if row == 0 and NUM_CLASSES:
            ax.set_title(CLASSES[yy[j]], fontsize=9)
plt.tight_layout(); plt.show()

## Stages 2–6 — Backbone → 3-D latent volume → 3-D SOM

`umtvit.models.UMTViT(cfg)` composes the dual-scale patch embed →
cross-attention → fusion → encoder (all L layers) → spatial uplifting into the
`H'×W'×L×C` voxel volume → contrastive projection head.
`Soft3DSOM.from_config(cfg)` builds the differentiable 3-D SOM the quantization
loss and hit/revival bookkeeping act on.

In [ ]:
# ── construct model + SOM ────────────────────────────────────────────────
set_seed(cfg.train.seed)
model = UMTViT(cfg).to(DEVICE)
som = Soft3DSOM.from_config(cfg).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
with torch.no_grad():
    probe_out = model(torch.zeros(1, cfg.dataset.channels,
                                  cfg.dataset.image_size, cfg.dataset.image_size,
                                  device=DEVICE))
print(f"UMT-ViT: {n_params/1e6:.3f} M params · "
      f"volume {tuple(probe_out['volume'].shape[1:])} · "
      f"proj {tuple(probe_out['proj'].shape[1:])} · "
      f"SOM {som.grid} ({som.K} neurons, update='{som.update}')")

## Training — self-supervised, label-free

`umtvit.engine.Trainer` runs the U4 loop (AdamW, warmup→cosine LR, the DESOM σ
anneal, all objective terms, dead-neuron revival). Its `on_epoch_end` callback
lets us snapshot — SOM weights, probe-image latent cubes, a pooled-embedding
subset, and SOM metrics — after every epoch (plus an untrained epoch-0
snapshot) without subclassing. These snapshots drive every animation below.

In [ ]:
# ── train via engine.Trainer, snapshotting each epoch ────────────────────
# fixed probe images (latent-cube tracking) + fixed embedding subset. Use the
# test split when it is large enough, else fall back to train (a tiny CI test
# split can be smaller than probe_images).
from torch.utils.data import Subset

probe_ds = eval_test_ds if len(eval_test_ds) >= VIZ["probe_images"] else eval_train_ds
n_probe = min(VIZ["probe_images"], len(probe_ds))
probe_x = torch.stack([probe_ds[i][0] for i in range(n_probe)])
probe_y = [probe_ds[i][1] for i in range(n_probe)]

_embed_rng = np.random.default_rng(cfg.train.seed)   # seeded subset (not first-N)
_n_embed = min(VIZ["embed_subset"], len(eval_train_ds))
_embed_idx = sorted(_embed_rng.permutation(len(eval_train_ds))[:_n_embed].tolist())
embed_loader = DataLoader(Subset(eval_train_ds, _embed_idx), batch_size=64)


@torch.no_grad()
def make_snapshot(trainer, epoch):
    """Mirror the canonical notebook's per-epoch snapshot dict."""
    mdl, s, dev = trainer.model, trainer.som, trainer.device
    was_training = mdl.training
    mdl.eval()
    vols = mdl(probe_x.to(dev))["volume"].to(torch.float16).cpu().numpy()
    feats, labels = [], []
    for x, y in embed_loader:
        feats.append(mdl(x.to(dev))["pooled"].cpu())
        labels.append(y)
    feats = torch.cat(feats)
    vox = torch.from_numpy(vols.astype(np.float32)).reshape(-1, s.weights.shape[1])
    snap = {
        "epoch": int(epoch),
        "som_weights": s.weights.detach().cpu().numpy().copy(),
        "probe_volumes": vols,
        "embeddings": feats.numpy(),
        "labels": torch.cat(labels).numpy(),
        "som_metrics": s.metrics(vox.to(dev)),
    }
    if was_training:
        mdl.train()
    return snap


snapshots = []


def on_epoch_end(epoch, metrics, trainer):
    snapshots.append(make_snapshot(trainer, epoch))
    mm = snapshots[-1]["som_metrics"]
    print(f"epoch {epoch:>3}/{trainer.budget_epochs} · loss {metrics['loss']:.3f} · "
          f"QE {mm['quantization_error']:.3f} · TE {mm['topographic_error']:.3f} · "
          f"dead {mm['dead_neuron_fraction']:.2f} · revived {metrics['revived']:>3} · "
          f"sigma {metrics['sigma']:.2f}")


trainer = Trainer(cfg, model, som, train_ds, on_epoch_end=on_epoch_end)
snapshots.append(make_snapshot(trainer, 0))            # untrained state (epoch 0)
t0 = time.time()
history = trainer.train()
# schedule constants for the SOM-animation title (σ trace).
SIGMA_START, SIGMA_END = trainer.sigma_start, trainer.sigma_end
TOTAL_STEPS, STEPS_PER_EPOCH = trainer.total_steps, trainer.steps_per_epoch


def sigma_at(step):
    return SIGMA_START * (SIGMA_END / SIGMA_START) ** (step / max(1, TOTAL_STEPS))


print(f"trained {trainer.step} steps in {time.time()-t0:.0f}s on {trainer.device}")

In [ ]:
# ── training curves (loss terms + SOM organization) ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for k in ("total", "ntxent", "som"):
    axes[0].plot(history[k], label=k)
axes[0].set_title("main losses"); axes[0].set_xlabel("step"); axes[0].legend()
for k in ("smooth", "order", "order_monotone"):
    axes[1].plot(history[k], label=k)
axes[1].set_title("topology regularizers"); axes[1].set_xlabel("step"); axes[1].legend()
ep = [s["epoch"] for s in snapshots]
for key, style in (("quantization_error", "-o"), ("topographic_error", "-s"),
                   ("dead_neuron_fraction", "-^")):
    axes[2].plot(ep, [s["som_metrics"][key] for s in snapshots], style,
                 label=key.replace("_", " "))
axes[2].set_title("SOM organization over training"); axes[2].set_xlabel("epoch")
axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## The latent cube — scroll through learned depth

Each probe image's encoder layers are uplifted into an `H'×W'×L×C` voxel
volume. The montage shows the channel-mean of each depth slice `z` (= encoder
layer); the animation glides continuously through that learned depth axis.

In [ ]:
# ── latent-cube slice montage ────────────────────────────────────────────
def slice_img(vol, l):
    """Channel-mean of depth slice l, per-slice min-max normalized."""
    s = vol[:, :, l, :].astype(np.float32).mean(-1)
    return (s - s.min()) / (np.ptp(s) + 1e-8)


final_vols = snapshots[-1]["probe_volumes"]
n_show = min(3, final_vols.shape[0])
L = final_vols.shape[3]
fig, axes = plt.subplots(n_show, L + 1, figsize=(1.6 * (L + 1), 1.8 * n_show),
                         squeeze=False)
for i in range(n_show):
    axes[i, 0].imshow(probe_x[i].permute(1, 2, 0).cpu().numpy())
    axes[i, 0].set_ylabel(CLASSES[probe_y[i]] if NUM_CLASSES else f"img {i}", fontsize=8)
    axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
    for l in range(L):
        axes[i, l + 1].imshow(slice_img(final_vols[i], l), cmap="magma")
        axes[i, l + 1].set_axis_off()
        if i == 0:
            axes[i, l + 1].set_title(f"z={l}", fontsize=9)
fig.suptitle("latent voxel volume — depth slices (z = encoder layer)", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── animation 1: gliding through the latent cube's depth axis ────────────
def cube_animation(vol, image, fps, max_frames):
    interp = max(2, min(10, max_frames // max(L - 1, 1)))
    zs = np.linspace(0, L - 1, (L - 1) * interp + 1)
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(6.4, 3.2))
    ax0.imshow(image); ax0.set_title("input", fontsize=9); ax0.set_axis_off()
    im = ax1.imshow(slice_img(vol, 0), cmap="magma", animated=True)
    ax1.set_axis_off()
    title = ax1.set_title("z = 0.0", fontsize=9)

    def frame(z):
        lo, hi = int(np.floor(z)), min(int(np.floor(z)) + 1, L - 1)
        a = z - np.floor(z)
        im.set_array((1 - a) * slice_img(vol, lo) + a * slice_img(vol, hi))
        title.set_text(f"latent depth z = {z:.1f} / {L - 1}")
        return [im, title]

    anim = animation.FuncAnimation(fig, frame, frames=zs, interval=1000 / fps, blit=False)
    plt.close(fig)
    return anim


anim1 = cube_animation(final_vols[0], probe_x[0].permute(1, 2, 0).cpu().numpy(),
                       VIZ["anim_fps"], VIZ["max_anim_frames"])
display(HTML(anim1.to_jshtml()))

## The SOM — a topographic atlas of visual structure

The 3-D SOM's U-matrix (neighbour-distance) reveals cluster boundaries per
grid z-layer; the hit maps show where voxels land. The second animation watches
the map organize over training (σ shrinking), the third watches the pooled
embeddings separate in a fixed PCA basis.

In [ ]:
# ── SOM maps (final state) ───────────────────────────────────────────────
GZ, GY, GX = som.grid


def umatrix_layers(weights):
    """[K,C] SOM weights -> U-matrix per z-layer of the neuron grid."""
    w = weights.reshape(GZ, GY, GX, -1)
    u = np.zeros((GZ, GY, GX))
    offs = ((1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0), (0, 0, 1), (0, 0, -1))
    for z in range(GZ):
        for y in range(GY):
            for x in range(GX):
                nb = [w[c] for c in ((z + dz, y + dy, x + dx) for dz, dy, dx in offs)
                      if 0 <= c[0] < GZ and 0 <= c[1] < GY and 0 <= c[2] < GX]
                u[z, y, x] = np.mean([np.linalg.norm(w[z, y, x] - n) for n in nb])
    return u


@torch.no_grad()
def bmu_hits():
    vox = torch.from_numpy(
        snapshots[-1]["probe_volumes"].astype(np.float32)
    ).reshape(-1, som.weights.shape[1]).to(som.weights.device)
    counts = torch.bincount(som.bmu(vox), minlength=som.K).cpu().numpy()
    return counts.reshape(GZ, GY, GX)


u_final, hits = umatrix_layers(snapshots[-1]["som_weights"]), bmu_hits()
fig, axes = plt.subplots(2, GZ, figsize=(2.1 * GZ, 4.4), squeeze=False)
for z in range(GZ):
    axes[0, z].imshow(u_final[z], cmap="viridis")
    axes[0, z].set_title(f"SOM z-layer {z}", fontsize=9); axes[0, z].set_axis_off()
    axes[1, z].imshow(hits[z], cmap="hot"); axes[1, z].set_axis_off()
axes[0, 0].set_ylabel("U-matrix"); axes[1, 0].set_ylabel("hits")
fig.suptitle("3-D SOM — U-matrix (top) and voxel hit maps (bottom) per grid layer")
plt.tight_layout(); plt.show()

In [ ]:
# ── animation 2: the SOM organizing itself over training ─────────────────
u_stack = [umatrix_layers(s["som_weights"]) for s in snapshots]
vmax = max(u.max() for u in u_stack) + 1e-8
fig, axes = plt.subplots(1, GZ, figsize=(2.1 * GZ, 2.5), squeeze=False)
ims = []
for z in range(GZ):
    ims.append(axes[0, z].imshow(u_stack[0][z], cmap="viridis", vmin=0, vmax=vmax,
                                 animated=True))
    axes[0, z].set_axis_off()
sup = fig.suptitle("SOM U-matrix — epoch 0")


def som_frame(i):
    for z in range(GZ):
        ims[z].set_array(u_stack[i][z])
    step = min(snapshots[i]["epoch"] * STEPS_PER_EPOCH, TOTAL_STEPS - 1)
    sup.set_text(f"SOM U-matrix — epoch {snapshots[i]['epoch']} (sigma={sigma_at(step):.2f})")
    return ims


anim2 = animation.FuncAnimation(fig, som_frame, frames=len(snapshots),
                                interval=700, blit=False)
plt.close(fig)
display(HTML(anim2.to_jshtml()))

In [ ]:
# ── animation 3: the embedding space taking shape (fixed PCA basis) ──────
last = snapshots[-1]["embeddings"]
mu = last.mean(0)
_, _, Vt = np.linalg.svd(last - mu, full_matrices=False)   # fixed PCA basis
proj_all = [(s["embeddings"] - mu) @ Vt[:2].T for s in snapshots]
lims = np.abs(np.concatenate(proj_all)).max() * 1.1
lab = snapshots[-1]["labels"]
tab10 = plt.cm.tab10
pt_colors = (tab10(np.asarray(lab) % 10) if NUM_CLASSES
             else np.tile(tab10(0), (len(lab), 1)))

fig, ax = plt.subplots(figsize=(5.2, 4.6))
sc = ax.scatter(*proj_all[0].T, c=pt_colors, s=14, alpha=0.8)
ax.set_xlim(-lims, lims); ax.set_ylim(-lims, lims)
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ttl = ax.set_title("pooled embeddings — epoch 0")
if NUM_CLASSES:
    handles = [Line2D([0], [0], marker="o", linestyle="", markersize=6,
                      markerfacecolor=tab10(i % 10), markeredgecolor="none",
                      label=CLASSES[i]) for i in range(NUM_CLASSES)]
    ax.legend(handles=handles, fontsize=8, loc="upper right")


def emb_frame(i):
    sc.set_offsets(proj_all[i])
    ttl.set_text(f"pooled embeddings (fixed PCA basis) — epoch {snapshots[i]['epoch']}")
    return [sc, ttl]


anim3 = animation.FuncAnimation(fig, emb_frame, frames=len(snapshots),
                                interval=700, blit=False)
plt.close(fig)
display(HTML(anim3.to_jshtml()))

## Did the Z-axis order by scale? (measured, not assumed)

For each depth slice we compute the spectral centroid of its 2-D power
spectrum. A centroid that **falls with depth** means shallow slices carry fine
detail and deep slices coarse structure — the imposed ordering bias took
effect. The *fair* per-channel centroid (from `umtvit.eval.zaxis_probe`) drives
the verdict; a rise is reported as an honest negative result.

In [ ]:
# ── Z-axis frequency probe (package eval.zaxis_probe) ────────────────────
zres = zaxis_probe(final_vols.astype(np.float32))
centroids = zres["per_channel_centroids"]                 # fair measure (drives verdict)
centroids_channel_mean = zres["channel_mean_centroids"]   # legacy channel-mean

# per-slice channel-mean spectra for display (radial_spectrum from the package)
spectra = []
for l in range(L):
    ps = [radial_spectrum(slice_img(final_vols[i], l))[1] for i in range(final_vols.shape[0])]
    spectra.append(np.mean(ps, axis=0))
centers = radial_spectrum(slice_img(final_vols[0], 0))[0]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 3.6))
for l, p in enumerate(spectra):
    ax0.plot(centers, p / (p.sum() + 1e-9), label=f"z={l}",
             color=plt.cm.plasma(l / max(L - 1, 1)))
ax0.set_xlabel("spatial frequency (cycles/px)"); ax0.set_ylabel("normalized power")
ax0.set_title("per-slice power spectra (channel-mean)"); ax0.legend(fontsize=8)
ax1.bar(range(L), centroids, color=plt.cm.plasma(np.linspace(0, 1, L)))
ax1.set_xlabel("depth slice z (encoder layer)")
ax1.set_ylabel("mean per-channel spectral centroid")
ax1.set_title(f"spectral centroid — {zres['verdict']}", fontsize=8)
plt.tight_layout(); plt.show()
print("spectral centroids by depth (mean per-channel):", np.round(centroids, 4))
print("spectral centroids by depth (channel-mean, old):", np.round(centroids_channel_mean, 4))
print("verdict:", zres["verdict"])

## Evaluation — labels enter here only

`umtvit.eval.run_evaluation` runs every §6 metric over the eval splits — frozen
linear probe + cosine k-NN (skipped in unlabeled mode), SOM quality, manifold
trustworthiness/continuity, and the Z-axis ordering probe — and returns one
nested dict. `render_report` turns it into the honest markdown run report.

In [ ]:
# ── run the full evaluation suite + render the report ────────────────────
results = run_evaluation(
    model, som, cfg, {"train": eval_train_ds, "test": eval_test_ds}, seed=cfg.train.seed)

probe, knn = results["linear_probe"], results["knn"]
som_eval, manifold = results["som"], results["manifold"]
tw = manifold["trustworthiness"]
chance = (probe or knn or {}).get("chance")
print(f"linear probe: {(probe or {}).get('accuracy')} · "
      f"k-NN: {(knn or {}).get('accuracy')} · chance: {chance}")
print(f"SOM  QE {som_eval['quantization_error']:.3f} · TE {som_eval['topographic_error']:.3f} · "
      f"dead {som_eval['dead_neuron_fraction']:.3f} · trustworthiness {tw}")

report_md = render_report(results)
display(HTML("<pre style='font-size:12px;line-height:1.35'>" + report_md + "</pre>"))

## Export artifacts

Writes `umtvit_artifacts/<name>/`: the probe latent cubes (npz + JSON sidecar),
initial+final SOM weights, the full `run_history.json` (config, loss history,
per-epoch SOM metrics, eval), and the package's `render_report` markdown.

In [ ]:
# ── write artifacts ──────────────────────────────────────────────────────
art_dir = Path("umtvit_artifacts") / cfg.dataset.name
art_dir.mkdir(parents=True, exist_ok=True)

# 1) latent cubes of the probe images (fp16) + JSON sidecar
cubes = final_vols.astype(np.float16)                      # [N,H',W',L,Cv]
np.savez_compressed(art_dir / "latent_cubes.npz", cubes=cubes,
                    labels=np.asarray(probe_y))
sidecar = {
    "array": "cubes", "dtype": "float16", "shape": list(cubes.shape),
    "axes": ["image", "H_prime", "W_prime", "Z", "C"],
    "axis_meaning": {
        "H_prime": "latent volume height (fusion grid)",
        "W_prime": "latent volume width (fusion grid)",
        "Z": "transformer depth == encoder layer index; a LEARNED hierarchy of "
             "representations, NOT physical/anatomical depth",
        "C": "per-slice latent channels (volume_channels)",
    },
    "labels": [int(v) for v in probe_y],
    "class_names": list(CLASSES) if NUM_CLASSES else None,
    "dataset": cfg.dataset.name,
}
(art_dir / "latent_cubes.json").write_text(json.dumps(sidecar, indent=2))

# 2) SOM weights — initial + final (fp16)
np.savez_compressed(art_dir / "som_weights.npz",
                    final=snapshots[-1]["som_weights"].astype(np.float16),
                    initial=snapshots[0]["som_weights"].astype(np.float16),
                    grid=np.asarray(som.grid))

# 3) full training history + per-epoch SOM metrics + eval
run_history = {
    "config": cfg.to_dict(),
    "history": {k: [float(x) for x in v] for k, v in history.items()},
    "per_epoch_som_metrics": [
        {"epoch": int(s["epoch"]), **{k: float(v) for k, v in s["som_metrics"].items()}}
        for s in snapshots],
    "eval": {
        "linear_probe_acc": (probe or {}).get("accuracy"),
        "knn_acc": (knn or {}).get("accuracy"),
        "chance": chance,
        "som_metrics": {k: float(v) for k, v in som_eval.items()},
        "trustworthiness": (None if tw is None or (isinstance(tw, float) and math.isnan(tw))
                            else float(tw)),
    },
    "spectral_centroids_by_depth": [float(c) for c in centroids],
    "spectral_centroids_channel_mean": [float(c) for c in centroids_channel_mean],
    "zaxis_verdict": zres["verdict"],
}
(art_dir / "run_history.json").write_text(json.dumps(run_history, indent=2))

# 4) human-readable report (the package's render_report)
(art_dir / "report.md").write_text(report_md)

print(f"artifacts written to {art_dir}/")
for f in sorted(art_dir.iterdir()):
    print(f"  {f.name:22s} {f.stat().st_size/1024:8.1f} KB")

## Export web bundle

Builds `umtvit_web.json` (schema v1) for the `/umtvit` explorer in the web app.
This matches `apps/web/src/lib/umtvit.ts` exactly — same builder as the
canonical notebook — so the drop-in validates field-for-field.

In [ ]:
# ── build umtvit_web.json (web explorer bundle, schema v1) ───────────────
import base64


def _finite(v):
    """JSON-safe float: None for NaN/inf (schema allows nulls for metrics)."""
    if v is None:
        return None
    v = float(v)
    return v if math.isfinite(v) else None


def _downsample_idx(n, cap=400):
    """Evenly-spaced <=cap indices over range(n), endpoints preserved."""
    if n <= cap:
        return list(range(n))
    return np.unique(np.linspace(0, n - 1, cap).round().astype(int)).tolist()


def _png_b64(chw, max_px=128):
    """[3,H,W] in [0,1] -> base64 PNG string, longest side <= max_px."""
    arr = (chw.clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    im = Image.fromarray(arr)
    if max(im.size) > max_px:
        scale = max_px / max(im.size)
        im = im.resize((max(1, round(im.width * scale)), max(1, round(im.height * scale))),
                       Image.BILINEAR)
    buf = io.BytesIO(); im.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")


def _cube(vol):
    """[H',W',L,Cv] volume -> [L][H'][W'] channel-mean, per-slice norm, 3dp."""
    return [np.round(slice_img(vol, l), 3).tolist() for l in range(vol.shape[2])]


_hidx = _downsample_idx(len(history["total"]))
_web_series = {
    k: ([float(history[k][i]) for i in _hidx] if history[k] else [])
    for k in ("total", "ntxent", "som", "smooth", "order")
}
init_vols = snapshots[0]["probe_volumes"]
web_probes = [
    {
        "label": (CLASSES[probe_y[i]] if (NUM_CLASSES and probe_y[i] >= 0) else None),
        "image_png_b64": _png_b64(probe_x[i]),
        "cube_initial": _cube(init_vols[i]),
        "cube_final": _cube(final_vols[i]),
    }
    for i in range(final_vols.shape[0])
]

web_bundle = {
    "version": 1,
    "dataset": {
        "name": cfg.dataset.name,
        "image_size": cfg.dataset.image_size,
        "augmentation": cfg.dataset.augmentation,
        "num_classes": int(NUM_CLASSES),
    },
    "model": {
        "dim": cfg.model.dim,
        "depth": cfg.model.depth,
        "volume_grid": cfg.model.volume_h,
        "volume_channels": cfg.model.volume_channels,
        "som_grid": list(cfg.model.som_grid),
        "cross_attention": cfg.model.cross_attention,
        "params_millions": round(n_params / 1e6, 3),
    },
    "metrics": {
        "linear_probe": _finite((probe or {}).get("accuracy")),
        "knn": _finite((knn or {}).get("accuracy")),
        "chance": _finite(chance),
        "som_quantization_error": _finite(som_eval["quantization_error"]),
        "som_topographic_error": _finite(som_eval["topographic_error"]),
        "som_dead_fraction": _finite(som_eval["dead_neuron_fraction"]),
        "trustworthiness": _finite(tw),
    },
    "spectral_centroids": [_finite(c) for c in centroids],
    "classes": list(CLASSES) if NUM_CLASSES else None,
    "history": {"steps": [int(i) for i in _hidx], "series": _web_series},
    "probes": web_probes,
    "som": {
        "grid": [int(GZ), int(GY), int(GX)],
        "epochs": [int(s["epoch"]) for s in snapshots],
        "umatrix": [np.round(u, 3).tolist() for u in u_stack],
        "hits_final": hits.astype(int).tolist(),
    },
    "embeddings": {
        "epochs": [int(s["epoch"]) for s in snapshots],
        "coords": [np.round(p, 2).tolist() for p in proj_all],
        "labels": ([int(v) for v in lab] if NUM_CLASSES else None),
    },
}

web_path = art_dir / "umtvit_web.json"
web_path.write_text(json.dumps(web_bundle, separators=(",", ":")))
_kb = web_path.stat().st_size / 1024
print(f"wrote {web_path} · {_kb:.1f} KB ({_kb/1024:.2f} MB)")
print("drop this file onto the /umtvit page of the web app")

## Optional — ablation study

`umtvit.engine.AblationRunner` trains the canonical `ABLATIONS` axes (no
cross-attention, full-pair, no-SOM, no-order, smooth-hwz, kohonen-EMA) on a
short shared budget and tabulates the §6 metrics side by side. Off by default
(it retrains the model once per axis); flip `RUN_ABLATIONS = True` to run it.

In [ ]:
# ── ablation comparison (behind a flag: retrains per axis) ───────────────
RUN_ABLATIONS = False

if RUN_ABLATIONS:
    def _train_data(c):
        return UniversalDataset(c, "train", "two_view")

    def _eval_data(c):
        return {"train": UniversalDataset(c, "train", "eval"),
                "test": UniversalDataset(c, "test", "eval")}

    # a short shared budget so every axis is quick (2 epochs of the base config)
    budget = {"train.epochs": min(2, cfg.train.epochs or 2), "train.max_steps": None}
    runner = AblationRunner(
        cfg, ABLATIONS, train_data=_train_data, eval_datasets=_eval_data,
        budget_overrides=budget, seed=cfg.train.seed)
    table = runner.run()
    print(table["table"])
else:
    print("RUN_ABLATIONS = False — skipping the ablation sweep "
          f"(set True to compare {[n for n, _ in ABLATIONS]}).")

## Conclusion

This notebook reproduces the canonical UMT-ViT experiment entirely through the
`umtvit` package: `load_config` → `UniversalDataset` → `UMTViT` / `Soft3DSOM` →
`engine.Trainer` → `eval.run_evaluation` / `render_report` → artifacts +
`umtvit_web.json`. Swapping datasets is a one-string change (`CONFIG_NAME`), and
the same code scales from the CPU `shapes` smoke config up to `ham10000` /
`eurosat` on a GPU. The Z-axis remains a *learned* hierarchy and the probe
numbers are SSL yardsticks — both reported as measured, never assumed.